# Model

## 1. Preliminary

Model - reasoning engine of agent

LLMs support:
- Tool calling - calling external tools (like databases queries or API calls) and use results in their responses.
- Structured output - where the model’s response is constrained to follow a defined format.
- Multimodality - process and return data other than text, such as images, audio, and video.
- Reasoning - models perform multi-step reasoning to arrive at a conclusion.

## 2. Configure model parameters

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

'''
init_chat_model(
  model: str | None = None,
  *,
  model_provider: str | None = None,
  configurable_fields: Literal['any'] | list[str] | tuple[str, ...] | None = None,
  config_prefix: str | None = None,
  **kwargs: Any = {}
) -> BaseChatModel | _ConfigurableModel

where:

**kwargs - Common parameters include:
    temperature: Model temperature for controlling randomness.
    max_tokens: Maximum number of output tokens.
    timeout: Maximum time (in seconds) to wait for a response.
    max_retries: Maximum number of retry attempts for failed requests.
    base_url: Custom API endpoint URL.
    rate_limiter: A BaseRateLimiter instance to control request rate.

'''

model = init_chat_model(
    model='openai/gpt-oss-120b',
    model_provider='groq',
    # thinking_level='medium'
)

C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
response = model.invoke("What is the capital of France?")
response

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'User asks: "What is the capital of France?" Straightforward answer: Paris. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 78, 'total_tokens': 117, 'completion_time': 0.081543627, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.022294217, 'prompt_tokens_details': None, 'queue_time': 0.285571041, 'total_time': 0.103837844}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_acbbc9c4c3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f02e5-bfa7-78c2-9175-802e3deb4fe7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 39, 'total_tokens': 117, 'output_token_details': {'reasoning': 21}})

In [3]:
agent = create_deep_agent( # bản chất là graph 
    model=model
)
type(agent)

langgraph.graph.state.CompiledStateGraph

Key methods:
* Invoke
* Stream
* Batch

### 2.1. Parameters

In [5]:
'''
- model: stringrequired

The name or identifier of the specific model you want to use with a provider. 
You can also specify both the model and its provider in a single argument using the ’:’ format, for example, ‘openai:o1’.

- api_key: string

The key required for authenticating with the model’s provider. This is usually issued when you sign up for access to the model. Often accessed by setting an environment variable.

- temperature: number

Controls the randomness of the model’s output. A higher number makes responses more creative; lower ones make them more deterministic.

- max_tokens: number

Limits the total number of tokens in the response, effectively controlling how long the output can be.

- timeout: number

The maximum time (in seconds) to wait for a response from the model before canceling the request.

- max_retries: number - default:"6"

The maximum number of attempts the system will make to resend a request if it fails due to issues like network timeouts or rate limits. 
Retries use exponential backoff with jitter. Network errors, rate limits (429), and server errors (5xx) are retried automatically. 
Client errors such as 401 (unauthorized) or 404 are not retried. For long-running agent tasks on unreliable networks, consider increasing this to 10–15.
'''

# Example with custom parameters:
model = init_chat_model(
    model="openai/gpt-oss-120b:free",
    model_provider="openrouter",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks -> kiểu bị lỗi mạng thì sẽ tự động retry lại, nếu mạng quá tệ thì có thể tăng số lần retry lên 10-15 lần
)

model = init_chat_model(
    "openrouter:openai/gpt-oss-120b:free",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=300,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks
)


In [14]:
# Testing models
model = init_chat_model(
    model="groq:openai/gpt-oss-120b",
    temperature=0.7,
    timeout=300,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks
)

## 3. Invocation

### 3.1. Invoke

In [15]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
# unit test
conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming.")
]
response = model.invoke(conversation)
print(response)

content='J’aime la programmation.' additional_kwargs={'reasoning_content': 'The user wants translation English to French. The phrase: "I love programming." => "J\'aime la programmation." Could also be "J\'adore la programmation." But literal translation: "J\'aime la programmation." Provide translation.'} response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 91, 'total_tokens': 151, 'completion_time': 0.123833491, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.003653762, 'prompt_tokens_details': None, 'queue_time': 0.289089238, 'total_time': 0.127487253}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ee95a-5344-7cc2-886f-38715b509826-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 91, 'output_tokens': 60, 'total_tokens': 151, 'output_token_details': {'reasoning': 46}}


In [16]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)

content="J'aime créer des applications." additional_kwargs={'reasoning_content': 'User wants translation to French. Provide translation: "J\'aime créer des applications." Could also be "J\'adore créer des applications." Probably "J\'aime développer des applications." But "building applications" could be "construire des applications". So translation: "J\'aime créer des applications." Provide that.'} response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 113, 'total_tokens': 190, 'completion_time': 0.162163271, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.004491921, 'prompt_tokens_details': None, 'queue_time': 0.314386739, 'total_time': 0.166655192}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6c36ac20de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ee95d-bd8f-7102-b5f9-cf1cd5391473-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 

In [6]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

KeyboardInterrupt: 

In [ ]:
conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming.")
]
response = model.invoke(conversation)
print(response)

### 3.2. Stream 

In [22]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||Par|rots|’| brilliant| plum|age| is| the| result| of| several| evolutionary| pressures| working| together|.| The| main| reasons| are|:

|###| |1|.| **|Sex|ual| selection|**
|-| **|Mate| attraction|:**| Bright|,| vivid| colors| are| a| reliable| signal| of| health| and| genetic| quality|.| In| many| par|rot| species|,| individuals| with| the| most| intense| coloration| are| preferred| as| mates|,| so| those| traits| get| amplified| over| generations|.
|-| **|Competition|:**| In| species| where| both| sexes| are| colorful|,| the| hue| and| pattern| can| also| signal| dominance| or| territorial| fitness| to| rivals|.

|###| |2|.| **|Species| and| individual| recognition|**
|-| **|Identity| tags|:**| In| dense| tropical| forests|,| many| par|rot| species| live| side|‑|by|‑|side|.| Dist|inct| color| patterns| help| birds| quickly| recognize| cons|pecific|s| (|members| of| the| same| species|)| an

In [23]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text,end="", flush=True)

Parrots are famous for their brilliant, multi‑colored plumage, and several evolutionary forces work together to produce and maintain those hues. The main reasons are:

### 1. **Sexual selection and social signaling**
- **Mate attraction:** Bright colors can signal good health, genetic quality, and the ability to find high‑quality food. In many bird species, individuals with more vivid plumage are preferred as mates.
- **Individual recognition:** Parrots live in complex social groups. Distinctive color patterns help birds recognize kin, mates, and rivals, reducing aggression and facilitating cooperation.

### 2. **Camouflage in a colorful environment**
- **Mimicry of the forest:** Tropical rainforests—the primary habitat of most parrots—are a mosaic of bright fruits, flowers, and dappled sunlight. A patchwork of reds, greens, blues, and yellows can break up a bird’s outline, making it harder for predators to spot.
- **Background matching:** Some species (e.g., the green macaw) blend wel

In [27]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text, end="\r", flush=True)

The sky is most commonly described as **blue** during a clear daytime. The blue color comes from Rayleigh scattering, where molecules in Earth’s atmosphere scatter shorter (blue) wavelengths of sunlight more than longer (red) wavelengths.

The sky is most commonly described as **blue** during a clear daytime. The blue color comes from Rayleigh scattering, where molecules in Earth’s atmosphere scatter shorter (blue) wavelengths of sunlight more than longer (red) wavelengths.

The sky is most commonly described as **blue** during a clear daytime. The blue color comes from Rayleigh scattering, where molecules in Earth’s atmosphere scatter shorter (blue) wavelengths of sunlight more than longer (red) wavelengths.

The sky is most commonly described as **blue** during a clear daytime. The blue color comes from Rayleigh scattering, where molecules in Earth’s atmosphere scatter shorter (blue) wavelengths of sunlight more than longer (red) wavelengths.

The sky is most commonly described as **

### 3.3. Batch

Batching a collection of independent requests to a model can significantly improve performance and reduce costs, as the processing can be done in parallel:

In [ ]:
# This section describes a chat model method batch(), which parallelizes model calls client-side -> return final output
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content='Parrots are famous for their brilliant reds, blues, greens, yellows, and oranges, and scientists think those vivid colors serve several purposes that help the birds survive and thrive in the wild. Here’s a rundown of the main reasons:\n\n### 1. **Sexual selection & mate choice**\n- **Showy “advertising”** – In many bird species, the brighter and more complex the plumage, the more attractive the individual is to potential mates. A vivid feather pattern can signal good health, good genetics, and the ability to find high‑quality food.\n- **Sexual dimorphism** – In some parrot species, males are more colorful than females (or have extra color patches) precisely because females use those cues when selecting a partner.\n\n### 2. **Species and individual recognition**\n- **Avoiding mix‑ups** – In dense tropical forests, many bird species look similar at a glance. Distinctive color patterns help parrots quickly recognize members of their own species, which is crucial for forming flock

In [29]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(0, AIMessage(content='Parrots’ brilliant plumage is the result of several evolutionary pressures working together—most notably sexual selection, species recognition, communication, and, to a lesser extent, camouflage. The colors we see arise from a mix of pigments and microscopic feather structures that manipulate light. Here’s a breakdown of why parrots are so colorful:\n\n| Factor | How it works | Why it matters for parrots |\n|--------|--------------|----------------------------|\n| **Sexual selection** | Bright colors can signal good health, good genes, or superior foraging ability. In many parrot species, both males and females are colorful, but in some the sexes differ (e.g., the male Eclectus parrot is bright green while the female is vivid red). | Individuals with more vivid plumage are often preferred as mates, so those traits become amplified over generations. |\n| **Species and individual recognition** | Specific color patterns (e.g., the red head of a Scarlet Macaw vs. the

# 4. Tool calling

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
- A function or coroutine to execute.

In [30]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


# 5. Structured output

LangChain supports multiple schema types and methods for enforcing structured output. 

LangChain support nhiều loại output: Pydantic, TypedDict, JSON Schema.

In [ ]:
'''
with_structured_output(
  self,
  schema: builtins.dict[str, Any] | type, 
  *,
  include_raw: bool = False,
  **kwargs: Any = {}
) -> Runnable[LanguageModelInput, builtins.dict[str, Any] | BaseModel]

where:
- Schema: A Pydantic model class or a dictionary representing the expected output structure. The model will validate the output against this schema.
- include_raw: Set include_raw=True to get both the parsed output and the raw AI message.
'''

Note: Định nghĩa sẵn một schema bằng dict của python (Pydantic, TypedDict hoặc JSON) để agent trả về

Method parameter: Some providers support different methods for structured output:
- 'json_schema': Uses dedicated structured output features offered by the provider.
- 'function_calling': Derives structured output by forcing a tool call that follows the given schema.
- 'json_mode': A precursor to 'json_schema' offered by some providers. Generates valid JSON, but the schema must be described in the prompt.

## 5.1. Pydantic 

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

## 5.2. TypedDict

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

## 5.3. JSON Schema 

In [ ]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

# 6. Advanced topics

## 6.1. Model profiles